# My Learning Journey: Attention Mechanisms

**Date**: January 27, 2026  
**Learning Goal**: Build the Self-Attention mechanism from scratch using PyTorch.

## What I'm Learning Today

I am moving into the heart of the Transformer architecture: **Attention**.
My goal is to understand how the model relates different words in a sequence to each other.

## My Understanding So Far

- **Embeddings** (Ch 2) turn tokens into fixed-size vectors.
- However, embeddings are static—the embedding for "bank" is the same in "river bank" and "bank account".
- **Attention** allows the model to contextually update these representations based on surrounding words.

## Plan

1.  **Simple Self-Attention**: Using loops (slow but clear).
2.  **Matrix Multiplication**: Vectorizing the operation (fast).
3.  **Scaled Dot-Product Attention**: Adding the scaling factor $1/\sqrt{d_k}$.
4.  **Causal Attention**: Masking future tokens (crucial for GPT).
5.  **Multi-Head Attention**: Running multiple attention mechanisms in parallel.

## Source Attribution

Concepts derived from *Build a Large Language Model From Scratch* (Raschka).  
Implementation is my own reconstruction to build intuition.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(123)
print(f"PyTorch version: {torch.__version__}")

## Section 3.3 — Simple Self-Attention (No Trainable Weights)

I'm starting with the most basic version: compute attention scores via dot products, normalize with softmax, then produce weighted context vectors. No learned weight matrices yet — just raw input embeddings talking to each other.

**Sentence**: "Your journey starts with one step" → 6 tokens → 6 embedding vectors (dim 3).

In [ ]:
# 6 token embeddings, each 3-dimensional
inputs = torch.tensor(
    [[0.43, 0.15, 0.89],  # "Your"    x^(1)
     [0.55, 0.87, 0.66],  # "journey" x^(2)
     [0.57, 0.85, 0.64],  # "starts"  x^(3)
     [0.22, 0.58, 0.33],  # "with"    x^(4)
     [0.77, 0.25, 0.10],  # "one"     x^(5)
     [0.05, 0.80, 0.55]]  # "step"    x^(6)
)
print("inputs shape:", inputs.shape)  # (6, 3)

In [ ]:
# Step 1: attention scores for query token x^(2) ("journey")
query = inputs[1]  # shape (3,)

# Dot product with every token in the sequence
attn_scores_2 = torch.zeros(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)  # higher = more similar

print("Unnormalized attention scores for x^(2):", attn_scores_2)

In [ ]:
# Step 2: normalize via softmax → attention weights sum to 1
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:"  , attn_weights_2)
print("Sum (should be 1.0):", attn_weights_2.sum().item())

# Step 3: context vector for x^(2) = weighted sum of all value vectors
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i

print("\nContext vector for 'journey':", context_vec_2)

In [ ]:
# Generalized: compute context vectors for ALL tokens at once using matrix ops
# inputs @ inputs.T gives all pairwise dot products: shape (6, 6)
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=-1)  # normalize along each row
all_context_vecs = attn_weights @ inputs            # weighted sum of values

print("All context vectors shape:", all_context_vecs.shape)   # (6, 3)
print("\nContext vector for x^(2) (should match above):", all_context_vecs[1])

## Section 3.4 — Trainable Weight Matrices

The simple attention above has a problem: the same input vectors are used as queries, keys, *and* values. Real transformers separate these roles via learned projection matrices:

$$Q = X W_Q, \quad K = X W_K, \quad V = X W_V$$

This gives the model freedom to learn how to "ask questions" (Q), "answer questions" (K), and "provide information" (V) differently.

**Dimensions**: `d_in=3` (input embedding), `d_out=2` (projected dimension).

In [ ]:
# Manually compute attention for x^(2) with raw parameter matrices
d_in, d_out = 3, 2

torch.manual_seed(123)
# nn.Parameter wraps a tensor so the optimizer will update it during training
W_query = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

x_2     = inputs[1]                  # query token: "journey"
query_2 = x_2 @ W_query              # shape (2,)
keys    = inputs @ W_key              # shape (6, 2)
values  = inputs @ W_value            # shape (6, 2)

# Scaled dot-product: divide by sqrt(d_k) to prevent vanishing gradients in softmax
attn_scores_2  = query_2 @ keys.T    # shape (6,)
attn_weights_2 = torch.softmax(attn_scores_2 / d_out**0.5, dim=-1)
context_vec_2  = attn_weights_2 @ values

print("context_vec_2:", context_vec_2)

In [ ]:
class SelfAttention_v1(nn.Module):
    """Self-attention using nn.Parameter (raw weight matrices, no bias)."""

    def __init__(self, d_in: int, d_out: int) -> None:
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        queries = x @ self.W_query
        keys    = x @ self.W_key
        values  = x @ self.W_value
        attn_scores  = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        return attn_weights @ values


class SelfAttention_v2(nn.Module):
    """Self-attention using nn.Linear (Kaiming weight init, no bias). Preferred."""

    def __init__(self, d_in: int, d_out: int, qkv_bias: bool = False) -> None:
        super().__init__()
        # nn.Linear stores weight as (d_out, d_in) internally; forward does x @ W.T
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)
        attn_scores  = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        return attn_weights @ values


torch.manual_seed(789)
sa_v1 = SelfAttention_v1(d_in=3, d_out=2)
out_v1 = sa_v1(inputs)
print("v1 context vecs (first row):", out_v1[0])

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in=3, d_out=2)
out_v2 = sa_v2(inputs)
print("v2 context vecs (first row):", out_v2[0])

## Section 3.5 — Causal (Masked) Attention

The attention I built above lets every token attend to every other token — including *future* tokens. That's wrong for a language model: when predicting word at position `t`, I shouldn't be able to see words at positions `t+1, t+2, …`.

The fix: mask the upper-triangular part of the attention score matrix with $-\infty$ before softmax. These become 0 after softmax (since $e^{-\infty} = 0$).

I also want to add dropout to the attention weights (regularization during training).

In [ ]:
# Show the mask step by step
context_length = inputs.shape[0]  # 6

# torch.triu(..., diagonal=1): upper triangle excluding main diagonal
# Entry (i,j)=1 means token j is "future" for token i → mask it out
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
print("Causal mask:\n", mask)

# Apply to attention scores for all tokens at once
queries = inputs @ sa_v2.W_query.weight.T
keys    = inputs @ sa_v2.W_key.weight.T
values  = inputs @ sa_v2.W_value.weight.T

attn_scores = queries @ keys.T
attn_scores_masked = attn_scores.masked_fill(mask.bool(), -torch.inf)

attn_weights_causal = torch.softmax(attn_scores_masked / keys.shape[-1]**0.5, dim=-1)
print("\nCausal attention weights (lower-triangular):\n", attn_weights_causal.detach())

In [ ]:
class CausalAttention(nn.Module):
    """Single-head causal self-attention — ready for batched (b, n, d_in) input."""

    def __init__(self, d_in: int, d_out: int, context_length: int,
                 dropout: float, qkv_bias: bool = False) -> None:
        super().__init__()
        self.d_out   = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # register_buffer: not a parameter, but travels to GPU with the model
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, n_tokens, d_in = x.shape
        queries = self.W_query(x)              # (b, n, d_out)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        # Batched dot-product: (b, n, d_out) @ (b, d_out, n) = (b, n, n)
        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:n_tokens, :n_tokens], -torch.inf
        )
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        return attn_weights @ values  # (b, n, d_out)


# Test with a 2-sample batch
batch = torch.stack((inputs, inputs), dim=0)  # (2, 6, 3)
torch.manual_seed(123)
ca = CausalAttention(d_in=3, d_out=2, context_length=6, dropout=0.0)
out_ca = ca(batch)
print("CausalAttention output shape:", out_ca.shape)   # (2, 6, 2)
print("First sample, first token:", out_ca[0, 0])

## Section 3.6 — Multi-Head Attention (MHA)

Running a single attention head means the model can only capture one type of relationship between tokens at a time. With multiple heads, each head can specialize — one might capture syntactic relationships, another semantic ones.

### 3.6.1 — Wrapper approach (intuitive)

Stack `num_heads` separate `CausalAttention` modules and concatenate their outputs along the embedding dimension.

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
    """Stack of CausalAttention heads; output = concatenation of all head outputs."""

    def __init__(self, d_in: int, d_out: int, context_length: int,
                 dropout: float, num_heads: int, qkv_bias: bool = False) -> None:
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Each head: (b, n, d_out); cat along last dim → (b, n, d_out * num_heads)
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)
# 2 heads, each outputs dim 2 → total output dim 4
mha_wrapper = MultiHeadAttentionWrapper(
    d_in=3, d_out=2, context_length=6, dropout=0.0, num_heads=2
)
out_wrapper = mha_wrapper(batch)
print("Wrapper output shape:", out_wrapper.shape)  # (2, 6, 4)

### 3.6.2 — Efficient MHA (weight splits, single forward pass)

Instead of running heads sequentially, project all heads at once and reshape.

**Key trick**: `view(b, n_tokens, num_heads, head_dim).transpose(1, 2)` → shape `(b, num_heads, n_tokens, head_dim)`. PyTorch then treats the `num_heads` dimension as a batch, computing all heads' attention scores in one matrix multiply.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Efficient MHA: one projection matrix per Q/K/V, split across heads via view/transpose."""

    def __init__(self, d_in: int, d_out: int, context_length: int,
                 dropout: float, num_heads: int, qkv_bias: bool = False) -> None:
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_out     = d_out
        self.num_heads = num_heads
        self.head_dim  = d_out // num_heads  # each head operates on this many dims

        self.W_query  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key    = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # final linear to mix head outputs
        self.dropout  = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, n_tokens, _ = x.shape

        # Project and reshape: (b, n, d_out) → (b, num_heads, n, head_dim)
        def split_heads(proj: torch.Tensor) -> torch.Tensor:
            return proj.view(b, n_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        queries = split_heads(self.W_query(x))
        keys    = split_heads(self.W_key(x))
        values  = split_heads(self.W_value(x))

        # All heads compute attention scores simultaneously: (b, num_heads, n, n)
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores.masked_fill_(self.mask.bool()[:n_tokens, :n_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Merge heads: (b, num_heads, n, head_dim) → (b, n, d_out)
        context = (attn_weights @ values).transpose(1, 2).contiguous()
        context = context.view(b, n_tokens, self.d_out)
        return self.out_proj(context)


torch.manual_seed(123)
mha = MultiHeadAttention(
    d_in=3, d_out=4, context_length=6, dropout=0.0, num_heads=2
)
out_mha = mha(batch)
print("MHA output shape:", out_mha.shape)   # (2, 6, 4)
print("First token context:", out_mha[0, 0].detach())

## My Chapter 3 Takeaways

| Step | What I learned |
|---|---|
| §3.3 Simple attention | Dot product captures similarity; softmax normalizes into a probability distribution |
| §3.4 Trainable weights | Separate Q/K/V projections give the model flexibility to learn different "roles" |
| §3.5 Causal masking | `torch.triu + masked_fill(-inf)` ensures no peeking at future tokens |
| §3.6 MHA Wrapper | Intuitive: run heads separately, concatenate — easy to understand |
| §3.6 Efficient MHA | Smart: single large matrix multiply + reshape — standard in real LLMs |

**What I still need to practice**:
- Working through the shape arithmetic by hand
- Integrating this MHA module into a full GPT block (Layer Norm + Feed Forward + residual connections — coming in Ch04)
- Understanding why `contiguous()` is needed after `transpose`